# Exploratory Data Analysis (EDA) - Matches

Este notebook deja el EDA de partidos mas alineado con la guia del taller: se parsea la fecha con el formato correcto, se renombra `as_` a `away_shots` y se construyen variables limpias para explorar volumen ofensivo, disciplina y mercado de apuestas.


In [ ]:
import os

# --- SOLUCION DE RUTAS (CWD) 100% PORTABLE ---
if os.path.exists('CSV (API)') and os.path.exists('EDA (Analisis Exploratorio de Datos)'):
    os.chdir('EDA (Analisis Exploratorio de Datos)')
elif os.path.exists('CSV (API)') and os.path.exists('EDA (Análisis Exploratorio de Datos)'):
    os.chdir('EDA (Análisis Exploratorio de Datos)')

print('Directorio configurado en:', os.getcwd())


In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px

warnings.filterwarnings('ignore')

df = pd.read_csv('../CSV (API)/matches.csv')
df = df.rename(columns={'as_': 'away_shots'})
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y', errors='coerce')
df['referee'] = df['referee'].fillna('Unknown')

odds_cols = ['implied_prob_h', 'implied_prob_d', 'implied_prob_a', 'b365h', 'b365d', 'b365a']
match_stat_cols = ['hs', 'away_shots', 'hf', 'af', 'hy', 'ay', 'hr', 'ar', 'hc', 'ac']
for col in odds_cols + match_stat_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

if 'implied_prob_h' not in df.columns:
    inv_sum = (1 / df['b365h']) + (1 / df['b365d']) + (1 / df['b365a'])
    df['implied_prob_h'] = (1 / df['b365h']) / inv_sum
    df['implied_prob_d'] = (1 / df['b365d']) / inv_sum
    df['implied_prob_a'] = (1 / df['b365a']) / inv_sum

df['goal_diff'] = df['fthg'] - df['ftag']
df['total_goals'] = df['fthg'] + df['ftag']
df['total_shots'] = df['hs'] + df['away_shots']
df['total_fouls'] = df['hf'] + df['af']
df['total_yellows'] = df['hy'] + df['ay']
df['total_reds'] = df['hr'] + df['ar']

print(f'Dataset cargado y limpio. Filas: {df.shape[0]}, Columnas: {df.shape[1]}')
df[['date', 'home_team', 'away_team', 'hs', 'away_shots', 'fthg', 'ftag', 'ftr']].head(3)


## 1. Visualizacion: volumen de tiros y diferencia de gol

Objetivo: revisar si dominar en volumen ofensivo se traduce en ventaja real en el marcador.


In [ ]:
fig1 = px.scatter_3d(
    df,
    x='hs',
    y='away_shots',
    z='goal_diff',
    color='ftr',
    color_discrete_map={'H': '#2ca02c', 'D': '#1f77b4', 'A': '#d62728'},
    labels={
        'hs': 'Tiros local',
        'away_shots': 'Tiros visitante',
        'goal_diff': 'Diferencia de goles',
        'ftr': 'Resultado'
    },
    hover_data=['home_team', 'away_team', 'fthg', 'ftag'],
    title='Volumen de tiros vs diferencia de goles'
)
fig1.update_traces(marker=dict(size=4, opacity=0.7, line=dict(width=0.5, color='black')))
fig1.show()


**Insight 1:** En general, los partidos donde el local tira bastante mas que el visitante tienden a empujar la diferencia de goles hacia terreno positivo. Aun asi, tambien aparecen excepciones importantes, que son justamente los partidos mas interesantes para el predictor.


## 2. Visualizacion: friccion del juego y disciplina

Objetivo: explorar si la intensidad fisica del partido cambia junto con el tipo de resultado.


In [ ]:
fig2 = px.scatter_3d(
    df,
    x='total_fouls',
    y='total_yellows',
    z='total_reds',
    color='ftr',
    color_discrete_map={'H': '#2ca02c', 'D': '#1f77b4', 'A': '#d62728'},
    labels={
        'total_fouls': 'Faltas totales',
        'total_yellows': 'Amarillas totales',
        'total_reds': 'Rojas totales',
        'ftr': 'Resultado'
    },
    hover_data=['home_team', 'away_team', 'referee'],
    title='Friccion del juego: faltas, amarillas y rojas'
)
fig2.update_traces(marker=dict(size=4, opacity=0.7, line=dict(width=0.5, color='gray')))
fig2.show()


**Insight 2:** La intensidad del partido no se traduce linealmente en tarjetas, pero los partidos con mas rojas tienden a volverse mucho mas caoticos. Eso refuerza la idea de que arbitros y disciplina pueden tener valor predictivo en el proyecto.


## 3. Visualizacion: mercado de apuestas vs goles reales

Objetivo: relacionar la expectativa del mercado con lo que realmente paso en el partido.


In [ ]:
fig3 = px.scatter_3d(
    df,
    x='implied_prob_h',
    y='implied_prob_a',
    z='total_goals',
    color='ftr',
    color_discrete_map={'H': '#2ca02c', 'D': '#1f77b4', 'A': '#d62728'},
    labels={
        'implied_prob_h': 'Probabilidad implicita local',
        'implied_prob_a': 'Probabilidad implicita visitante',
        'total_goals': 'Goles totales',
        'ftr': 'Resultado'
    },
    hover_data=['home_team', 'away_team', 'b365h', 'b365a'],
    title='Mercado de apuestas vs goles reales'
)
fig3.update_traces(marker=dict(size=4, opacity=0.65, line=dict(width=0.5, color='gray')))
fig3.show()


**Insight 3:** El mercado separa bien los favoritismos, pero no garantiza marcadores abultados. Hay muchos partidos donde un claro favorito gana con pocos goles, y tambien varios donde el volumen total de goles no sigue la intuicion inicial de las cuotas.
